# Лабораторная 3. Данные приехали, но верить им пока рано

Инженер построил отчёт по выручке, аналитик сделал дашборд, всё красиво.
Через месяц приходит бухгалтерия: «у вас выручка за март на четыре процента
больше нашей».

Разбираются неделю. Выясняется, что один и тот же отзыв приехал из сервиса
дважды и при соединении задвоил часть заказов. Что у восьми заказов статус
«доставлен», а даты доставки нет, и эти заказы выпали из расчёта срока
доставки – средний срок получился красивее настоящего. Что в платежах есть
строки с нулевой суммой и типом `not_defined`, которые никто не должен был
считать выручкой.

Ни одна из этих проблем не выглядит как ошибка в коде. Код отработал
безупречно: он посчитал ровно то, что ему дали. Разбираться будет не он,
а живой человек, у которого спросят про четыре процента. Разница между «данные
загрузились» и «данным можно верить» – содержание этой работы.

## Перед началом: новые слова

| Термин | Простыми словами |
|---|---|
| **ETL и ELT** | Две схемы работы: преобразовать данные по дороге в хранилище или сначала загрузить, а преобразовать уже внутри. |
| **Проверки качества – data quality checks** | Правила, которым данные обязаны соответствовать, записанные кодом и выполняемые при каждом запуске. |
| **Карантин – quarantine** | Отдельное место для строк, не прошедших проверку, вместе с причиной отбраковки. |
| **Контракт данных – data contract** | Договорённость с поставщиком: какие колонки, каких типов, с какими значениями и когда приезжают. |
| **Развитие схемы – schema evolution** | Изменения структуры источника со временем: новые колонки, новые типы, переименования. |
| **Персональные данные – PII** | Сведения, по которым можно узнать конкретного человека: имя, телефон, адрес, платёжные реквизиты. |

## Подготовка

Ячейка готовит окружение и читает данные, которые вы подготовили
во второй работе.

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
import numpy as np
import pandas as pd
import paths
from dq import CRITICAL, WARNING, DataQualityError, Suite, save_reports
from lab_utils import require_artifact, save_table
from stages import ensure_stage

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# Зависимость от ЛР 2: в обычном прохождении это ваш артефакт. В новой
# Colab-сессии восстанавливаем только предыдущий слой RAW/Parquet, не CLEAN.
ensure_stage("parquet")
require_artifact(paths.PARQUET / "olist_orders_dataset.parquet", "ЛР 2")

raw_orders = pd.read_parquet(paths.PARQUET / "olist_orders_dataset.parquet")
raw_items = pd.read_parquet(paths.PARQUET / "olist_order_items_dataset.parquet")
raw_payments = pd.read_parquet(paths.PARQUET / "olist_order_payments_dataset.parquet")
raw_reviews = pd.read_parquet(paths.PARQUET / "olist_order_reviews_dataset.parquet")
raw_customers = pd.read_parquet(paths.PARQUET / "olist_customers_dataset.parquet")
raw_products = pd.read_parquet(paths.PARQUET / "olist_products_dataset.parquet")
raw_sellers = pd.read_parquet(paths.PARQUET / "olist_sellers_dataset.parquet")

print(f"raw_orders   : {len(raw_orders):>7,}")
print(f"raw_items    : {len(raw_items):>7,}")
print(f"raw_payments : {len(raw_payments):>7,}")
print(f"raw_reviews  : {len(raw_reviews):>7,}")

In [ ]:
from course_visuals import show_visual

show_visual("quality")

## Где происходит преобразование

**ETL** – Extract → Transform → Load. Данные забирают из источника,
преобразуют по дороге (в Python, в Spark, в специальном ETL-инструменте)
и кладут в хранилище уже готовыми.

**ELT** – Extract → Load → Transform. Данные сначала кладут в хранилище
как есть, а преобразования выполняют уже внутри него – обычно SQL-запросами.

```
  ETL:  источник ──→ [ преобразование ] ──→ хранилище (только готовое)
  ELT:  источник ──→ хранилище (RAW) ──→ [ преобразование внутри ] ──→ CORE, MART
```

| | ETL | ELT |
|---|-----|-----|
| Где считаем | отдельный процесс/кластер | внутри хранилища |
| Что лежит в хранилище | только обработанное | и сырое, и обработанное |
| Пересчитать историю | заново выкачивать из источника | пересчитать SQL по RAW |
| Кто может менять логику | инженер | ещё и аналитик (SQL) |
| Когда лучше | тяжёлые преобразования, нетабличные данные, ограничения по PII | современные аналитические хранилища, где вычисления дёшевы |

ELT стал нормой, когда хранилища подешевели и научились быстро считать:
проще сложить всё сырое и пересчитывать, чем каждый раз ходить в источник.
Сырой слой у нас уже есть, и сейчас мы делаем первое преобразование –
слой **CLEAN**.

## О чём мы договариваемся с поставщиком данных

CSV не содержит описания колонок, поэтому каждый читатель угадывает типы сам –
и угадывает по-разному. Один прочитает почтовый индекс числом и потеряет
ведущий ноль, другой – строкой. Дальше эти две таблицы соединяют по индексу,
и ключи не совпадают.

Лечится это тем, что типы задают явно, а не полагаются на автоопределение.
Описание колонок и их типов называют **схемой (schema)**, а договорённость
с поставщиком о том, что именно и когда он присылает, – **контрактом данных
(data contract)**.

Контракт не обязан быть формальным документом. Минимальный рабочий вариант
звучит так: вот таблица; `order_id` – уникальный ключ; `price` – положительное
число; `order_status` принимает пять значений; данные приезжают к шести утра;
об изменениях предупреждаем за две недели. Ценность появляется, когда это
записано и проверяется при каждой загрузке.

### Что происходит, когда схема меняется

Источник живёт своей жизнью, и структура выгрузки меняется. Разберём три
понедельника подряд.

В понедельник `price` приезжает числом, всё работает. Во вторник поставщик
добавляет колонку `currency`. Это совместимое изменение: наш расчёт её не
читает и продолжает работать – при условии, что мы берём колонки по именам,
а не по порядковым номерам. В пятницу `price` начинает приезжать строкой
`"1 299,00"`. Вот это ломает всё: суммирование текста либо упадёт, либо –
что хуже – молча склеит строки.

Изменения структуры источника со временем называют **развитием схемы (schema
evolution)**. Добавление необязательной колонки обычно безопасно. Смена типа
или смысла существующей колонки несовместима с тем, что уже работает, и
замечать её нужно на входе, а не по жалобе на дашборд.

### Кому какие колонки нужны

Модели, предсказывающей задержку доставки, нужны город, категория товара
и история заказов. Имя покупателя и номер его телефона ей не нужны – качество
прогноза от них не изменится.

Сведения, по которым можно узнать конкретного человека, называют
**персональными данными (PII)**. В нашем наборе их нет: он опубликован
обезличенным, покупатель описан случайным идентификатором, городом, штатом
и первыми цифрами почтового индекса. В рабочей системе такие колонки обычно
есть, и к ним применяют два правила: доступ выдают под задачу, а не «на всякий
случай» (принцип наименьших привилегий, least privilege), и идентификаторы
заменяют на псевдонимы там, где настоящие значения не нужны. Возможность
прочитать таблицу и необходимость её читать – разные вещи.

In [ ]:
# Посмотрим на типы, которые pandas «угадал» из Parquet (а туда они попали из CSV).
pd.DataFrame({
    "тип": raw_orders.dtypes.astype(str),
    "пропусков": raw_orders.isna().sum(),
    "% пропусков": (raw_orders.isna().mean() * 100).round(2),
})

## Как данные ломаются на практике

| Класс | Пример у нас | Чем опасен |
|-------|--------------|-----------|
| **Типы** | дата приехала строкой | сравнения и сортировки врут |
| **Пропуски** | нет даты доставки у доставленного заказа | метрика считается по части данных |
| **Дубликаты** | один `review_id` на два заказа | задвоенные суммы после JOIN |
| **Нарушение бизнес-правил** | платёж на 0 с типом `not_defined`; доставка раньше покупки | «невозможные» строки в отчёте |

### Про NULL отдельно

Пропуск сам по себе не дефект. У него есть смысл, и этот смысл надо назвать:

* `order_delivered_customer_date` пуста у отменённого заказа – **это норма**,
  заказ не доставляли;
* та же колонка пуста у заказа со статусом `delivered` – **это дефект**,
  мы не знаем, что произошло на самом деле;
* `product_category_name` пуста – **это неизвестное значение**, его можно
  заменить на явное `unknown`, но нельзя молча выбросить товар из отчёта.

Правило: сначала объясните смысл пропуска, потом решайте, что с ним делать.
`dropna()` без объяснения – это не очистка, а потеря данных.

In [ ]:
# Ищем дефекты своими руками – прежде чем описывать их правилами.
o = raw_orders.copy()
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    o[c] = pd.to_datetime(o[c])

delivered = o["order_status"] == "delivered"
delivered_at = o["order_delivered_customer_date"]

checks = [
    ("статус «доставлен», но даты доставки нет", delivered & delivered_at.isna()),
    ("статус не «доставлен», а дата доставки есть", ~delivered & delivered_at.notna()),
    ("доставлен раньше, чем куплен", delivered_at < o["order_purchase_timestamp"]),
    ("подтверждён раньше, чем куплен",
     o["order_approved_at"] < o["order_purchase_timestamp"]),
    ("один review_id встречается дважды", raw_reviews["review_id"].duplicated()),
    ("у одного заказа больше одного отзыва", raw_reviews["order_id"].duplicated()),
    ("платёж с неположительной суммой", raw_payments["payment_value"] <= 0),
    ("полный дубликат строки в платежах", raw_payments.duplicated()),
    ("позиция с неположительной ценой", raw_items["price"] <= 0),
    ("позиция с отрицательной доставкой", raw_items["freight_value"] < 0),
    ("товар без категории", raw_products["product_category_name"].isna()),
]

findings = {name: int(mask.sum()) for name, mask in checks}
pd.Series(findings, name="найдено строк").to_frame()

Часть счётчиков может оказаться нулевой – это нормально: набор дефектов
зависит от того, какая версия данных у вас скачалась. Правила мы всё равно
описываем все: проверка, которая сегодня ничего не нашла, завтра поймает
изменившийся источник. **Проверки пишут не под текущие данные, а под контракт.**

In [ ]:
# Смотрим на дубликаты отзывов вблизи – это ключевой сюжет лабораторной.
dups = raw_reviews[raw_reviews["review_id"].duplicated(keep=False)].sort_values("review_id")
print(f"строк с повторяющимся review_id: {len(dups)}")
cols = ["review_id", "order_id", "review_score", "review_answer_timestamp"]
dups[cols].head(6)

Посмотрите внимательно: `review_id` один и тот же, а `order_id` – **разные**.

Это не техническая копия строки. Это особенность источника: один отзыв
оказался привязан к двум заказам. И теперь всё зависит от того, какое зерно
нам нужен:

* если таблица «один отзыв» – это дубликат, и лишнюю строку надо убрать;
* если таблица «отзыв по заказу» – это **две разные строки**, и удалять
  ничего нельзя, иначе мы потеряем заказ.

Отсюда главное практическое правило этой работы:

> **Дубликат – это не «одинаковые строки». Дубликат – это несколько строк
> на один экземпляр того, что описывает зерно таблицы.**

Пока вы не назвали зерно, слово «дубликат» не имеет смысла.

## Плохую строку нельзя просто выбросить

Что делать со строкой, которая нарушает правило? Есть четыре стратегии,
и выбор – инженерное решение, а не вкусовщина:

| Стратегия | Когда уместна | Риск |
|-----------|---------------|------|
| Починить (нормализовать) | понятно, как правильно: регистр, пробелы, формат даты | можно «починить» неверно |
| Отправить в карантин | строка нарушает критическое правило, но остальные данные нужны | про карантин забывают и не смотрят |
| Уронить весь запуск | нарушен контракт целиком: нет колонки, сломался формат | простой системы и разбор ночью |
| Пропустить и записать в warning | дефект некритичен для витрины | накопление мусора |

**Карантин** – отдельная таблица, куда мы складываем отбракованные строки
вместе с причиной отбраковки. Он честнее, чем `dropna()`, по трём причинам:
строки не исчезают бесследно; можно посчитать, сколько данных мы теряем;
можно вернуть их обратно, когда источник починят.

```
           ┌── проходит правила ──→ CLEAN ──→ витрины
  RAW ──→ ─┤
           └── нарушает правила ──→ QUARANTINE (+ причина) ──→ разбор
```

In [ ]:
# Ожидания от RAW-таблицы, записанные кодом. Сейчас они не пройдут – так и надо.
raw_suite = (Suite("raw_orders")
             .not_null("order_id")
             .unique("order_id")
             .in_set("order_status", {"delivered", "shipped", "canceled",
                                      "unavailable", "invoiced", "processing",
                                      "created", "approved"})
             .dtype("order_purchase_timestamp", "datetime", severity=WARNING)
             .custom("статус delivered ⇒ есть дата доставки",
                     lambda df: (df["order_status"] == "delivered")
                     & df["order_delivered_customer_date"].isna()))

raw_suite.run(raw_orders).show()

Видно две разные вещи: `order_id` в порядке (ключ уникален), а вот тип даты
и бизнес-правило – нет. Их и исправляет слой CLEAN.

## Что ломается в этом месте

**Соединили заказы с отзывами, не разобравшись с дубликатами.**

In [ ]:
naive = raw_orders.merge(raw_reviews[["order_id", "review_score"]],
                         on="order_id", how="left")
print(f"заказов                        : {len(raw_orders):,}")
print(f"строк после join с отзывами    : {len(naive):,}")
print(f"лишних строк                   : {len(naive) - len(raw_orders):,}")
print(f"средняя оценка по такому join  : {naive['review_score'].mean():.4f}")

one_per_order = (raw_reviews.sort_values("review_answer_timestamp")
                 .drop_duplicates(subset="order_id", keep="last"))
print(f"средняя оценка при grain «отзыв на заказ»: "
      f"{raw_orders.merge(one_per_order[['order_id', 'review_score']], on='order_id', how='left')['review_score'].mean():.4f}")
print("\nРазница в третьем знаке кажется мелочью – пока кто-нибудь не построит")
print("на этой метрике KPI поддержки и не начнёт принимать решения.")

**Удалили подозрительные строки молча.**

In [ ]:
bad_mask = ((raw_orders["order_status"] == "delivered")
            & raw_orders["order_delivered_customer_date"].isna())
print(f"строк нарушает правило: {int(bad_mask.sum())}")
print("dropna-подход  : строки исчезли, никто не знает сколько и почему")
print("карантин       : строки лежат отдельно с причиной, их можно посчитать,")
print("                 показать поставщику данных и вернуть после починки")
raw_orders.loc[bad_mask, ["order_id", "order_status",
                          "order_purchase_timestamp",
                          "order_delivered_customer_date"]].head(3)

**Тип колонки «почти правильный».**

In [ ]:
partner = pd.DataFrame({
    "order_id": ["a1", "a2", "a3"],
    "price": ["1 200,00", "350,50", "89,90"],      # партнёр прислал с запятой
})
print("тип колонки price:", partner["price"].dtype)
print("сумма «как есть» :", partner["price"].sum(), "  ← это конкатенация строк!")

fixed = partner.assign(
    price=partner["price"].str.replace(" ", "", regex=False)
                          .str.replace(",", ".", regex=False)
                          .astype(float))
print("сумма после приведения типа:", fixed["price"].sum())
print("\nСамое опасное здесь – что первый вариант не упал с ошибкой.")
print("Он честно вернул ответ. Просто неправильный.")

## Строим слой CLEAN

Мы собираем слой **CLEAN**: таблицы с явными типами, известным зерном
и проверенными бизнес-правилами. Всё, что не проходит критические правила,
уходит в карантин `data/quality/`.

### Задание 1. `clean_orders`

Контракт таблицы:

| колонка | тип | правило |
|---------|-----|---------|
| `order_id` | строка | уникален, без пропусков – это зерно таблицы |
| `customer_id` | строка | без пропусков |
| `order_status` | строка | одно из значений `ORDER_STATUSES` (см. код) |
| `order_purchase_timestamp` | datetime | без пропусков |
| `order_approved_at` | datetime | пропуск допустим |
| `order_delivered_carrier_date` | datetime | пропуск допустим |
| `order_delivered_customer_date` | datetime | пропуск допустим только у недоставленных |
| `order_estimated_delivery_date` | datetime | без пропусков |
| `is_delivered` | bool | `order_status == 'delivered'` |
| `delivery_days` | целое (с пропусками) | полных суток от покупки до доставки; `NaN`, если не доставлен |
| `is_late` | bool с пропусками | доставлен **позже** `order_estimated_delivery_date`; `NA`, если не доставлен |

**Критические правила – нарушители уходят в карантин, а не в CLEAN:**

* `R1` статус `delivered`, но `order_delivered_customer_date` пуст;
* `R2` статус не `delivered`, а дата доставки заполнена;
* `R3` дата доставки раньше даты покупки;
* `R4` дата подтверждения раньше даты покупки.

Карантин сохраните в `data/quality/quarantine_orders.parquet`, добавив колонку
`reason` с кодом правила (`R1`…`R4`; если строка нарушает несколько правил,
достаточно любого одного из них).

Подумайте: почему `delivery_days` нельзя считать до того, как вы отправили
в карантин строки по правилу `R3`?

In [ ]:
ORDER_STATUSES = {"delivered", "shipped", "canceled", "unavailable",
                  "invoiced", "processing", "created", "approved"}
DATE_COLS = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]


def build_clean_orders(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Возвращает (clean_orders, quarantine_orders)."""
    # TODO:
    #  1. привести пять колонок DATE_COLS к datetime;
    #  2. посчитать маски нарушений R1..R4 и собрать карантин с колонкой reason;
    #  3. оставить в clean только строки, не нарушившие критические правила;
    #  4. добавить is_delivered, delivery_days, is_late.
    raise NotImplementedError("реализуйте build_clean_orders")


clean_orders, quarantine_orders = build_clean_orders(raw_orders)
print(f"clean_orders      : {len(clean_orders):,}")
print(f"quarantine_orders : {len(quarantine_orders):,}")
if len(quarantine_orders):
    print(quarantine_orders["reason"].value_counts().to_dict())
clean_orders.head(3)

In [ ]:
from checks.lab03 import (check_task_1, check_task_2, check_task_3, lab03_result)

save_table(clean_orders, paths.CLEAN, "clean_orders")
quarantine_orders.to_parquet(paths.QUALITY / "quarantine_orders.parquet", index=False)
check_task_1(clean_orders, quarantine_orders, raw_orders)

### Задание 2. Найдите и исправьте ошибку в готовой функции

Функция `build_clean_payments` ниже написана вашим коллегой. Она выглядит
разумно, запускается без ошибок и даже проходит проверку «нет дубликатов».
Но она **неверна**.

Контракт таблицы `clean_payments`:

| | |
|---|---|
| зерно | одна строка = **одна платёжная операция**, ключ `(order_id, payment_sequential)` |
| `payment_value` | `float`, строго больше нуля |
| `payment_installments` | целое ≥ 1 |
| `payment_type` | одно из `{credit_card, boleto, voucher, debit_card}` |
| инвариант | сумма `payment_value` по CLEAN = сумма по RAW минус сумма отправленного в карантин |

Полные дубликаты строк (все колонки совпадают) – это техническая копия,
их надо убрать. Строки с неположительной суммой или с типом `not_defined` –
в карантин `data/quality/quarantine_payments.parquet` с колонкой `reason`.

**Задание:** найдите ошибку, объясните её в ячейке-ответе и исправьте функцию.
Подсказка: начните с вопроса «какое зерно у таблицы платежей?».

In [ ]:
PAYMENT_TYPES = {"credit_card", "boleto", "voucher", "debit_card"}


def build_clean_payments(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Возвращает (clean_payments, quarantine_payments)."""
    out = df.copy()

    # ↓↓↓ здесь ошибка коллеги – исправьте её ↓↓↓
    out = out.drop_duplicates(subset=["order_id"], keep="first")
    # ↑↑↑ ---------------------------------- ↑↑↑

    out["payment_value"] = out["payment_value"].astype(float)
    out["payment_installments"] = out["payment_installments"].astype("int64")

    bad_value = out["payment_value"] <= 0
    bad_type = ~out["payment_type"].isin(PAYMENT_TYPES)
    reason = pd.Series(pd.NA, index=out.index, dtype="object")
    reason = reason.mask(bad_type, "недопустимый payment_type")
    reason = reason.mask(bad_value, "payment_value <= 0")

    bad = reason.notna()
    quarantine = out[bad].copy()
    quarantine["reason"] = reason[bad]
    clean = out[~bad].copy()
    return clean.reset_index(drop=True), quarantine.reset_index(drop=True)


clean_payments, quarantine_payments = build_clean_payments(raw_payments)
print(f"raw_payments      : {len(raw_payments):,}")
print(f"clean_payments    : {len(clean_payments):,}")
print(f"quarantine        : {len(quarantine_payments):,}")
print(f"сумма RAW         : {raw_payments['payment_value'].sum():,.2f}")
print(f"сумма CLEAN       : {clean_payments['payment_value'].sum():,.2f}")

**Ваш ответ:** в чём именно была ошибка и к каким последствиям она приводила?

*(напишите здесь 2–4 предложения)*

In [ ]:
save_table(clean_payments, paths.CLEAN, "clean_payments")
quarantine_payments.to_parquet(paths.QUALITY / "quarantine_payments.parquet", index=False)
check_task_2(clean_payments, quarantine_payments, raw_payments)

### Задание 3. `clean_reviews` – решите, что здесь дубликат

Контракт таблицы `clean_reviews`:

| | |
|---|---|
| зерно | **один отзыв на заказ**: `order_id` уникален |
| колонки | `review_id`, `order_id`, `review_score`, `review_creation_date`, `review_answer_timestamp` |
| `review_score` | целое от 1 до 5, без пропусков |
| даты | тип datetime |
| правило разрешения конфликтов | если у заказа несколько отзывов – оставить **последний** по `review_answer_timestamp`; при равенстве – с меньшим `review_id` |
| инвариант | ни один заказ, у которого был отзыв, не должен потеряться |

Перед тем как писать код, ответьте себе на вопрос: строки с одинаковым
`review_id`, но разными `order_id` – это дубликаты при таком зерне или нет?
От ответа зависит, сколько строк останется.

In [ ]:
def build_clean_reviews(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: приведите типы, приведите таблицу к grain «один отзыв на заказ»
    #       и не потеряйте заказы.
    raise NotImplementedError("реализуйте build_clean_reviews")


clean_reviews = build_clean_reviews(raw_reviews)
print(f"raw_reviews   : {len(raw_reviews):,} строк, "
      f"{raw_reviews['order_id'].nunique():,} заказов")
print(f"clean_reviews : {len(clean_reviews):,} строк, "
      f"{clean_reviews['order_id'].nunique():,} заказов")
clean_reviews.head(3)

**Ваш ответ:** почему дедупликация по `review_id` была бы здесь ошибкой?
Сколько заказов вы бы потеряли?

*(напишите здесь 2–4 предложения)*

In [ ]:
save_table(clean_reviews, paths.CLEAN, "clean_reviews")
check_task_3(clean_reviews, raw_reviews)

## Остальные таблицы

Позиции заказа, клиентов, товары и продавцов приводим в порядок по тем же
принципам. Здесь два приёма: нормализация строк – регистр и лишние пробелы
дают «два разных города с одинаковым названием» – и явная замена неизвестной
категории на `unknown` вместо удаления товара.

In [ ]:
def build_clean_items(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    out["shipping_limit_date"] = pd.to_datetime(out["shipping_limit_date"], errors="coerce")
    out["price"] = out["price"].astype(float)
    out["freight_value"] = out["freight_value"].astype(float)

    reason = pd.Series(pd.NA, index=out.index, dtype="object")
    reason = reason.mask(out["freight_value"] < 0, "freight_value < 0")
    reason = reason.mask(out["price"] <= 0, "price <= 0")
    bad = reason.notna()
    q = out[bad].copy()
    q["reason"] = reason[bad]
    return out[~bad].reset_index(drop=True), q.reset_index(drop=True)


def build_clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["customer_city"] = out["customer_city"].astype(str).str.strip().str.lower()
    out["customer_state"] = out["customer_state"].astype(str).str.strip().str.upper()
    out["customer_zip_code_prefix"] = out["customer_zip_code_prefix"].astype("int64")
    return out.drop_duplicates(subset="customer_id").reset_index(drop=True)


def build_clean_sellers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["seller_city"] = out["seller_city"].astype(str).str.strip().str.lower()
    out["seller_state"] = out["seller_state"].astype(str).str.strip().str.upper()
    return out.drop_duplicates(subset="seller_id").reset_index(drop=True)


def build_clean_products(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["product_category_name"] = (out["product_category_name"]
                                    .astype("object").fillna("unknown")
                                    .astype(str).str.strip().str.lower())
    for c in ["product_weight_g", "product_length_cm",
              "product_height_cm", "product_width_cm"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.drop_duplicates(subset="product_id").reset_index(drop=True)


clean_items, quarantine_items = build_clean_items(raw_items)
clean_customers = build_clean_customers(raw_customers)
clean_sellers = build_clean_sellers(raw_sellers)
clean_products = build_clean_products(raw_products)

for name, df in [("clean_items", clean_items), ("clean_customers", clean_customers),
                 ("clean_sellers", clean_sellers), ("clean_products", clean_products)]:
    save_table(df, paths.CLEAN, name)
    print(f"{name:<16}: {len(df):>7,} строк")
quarantine_items.to_parquet(paths.QUALITY / "quarantine_items.parquet", index=False)
print(f"{'в карантине':<16}: {len(quarantine_items):>7,} позиций")

## Финальная валидация слоя CLEAN

Теперь то, ради чего всё затевалось: набор ожиданий, который выполняется
при каждом запуске. В пятой работе этот же код станет отдельным шагом
обработки и получит право её останавливать.

In [ ]:
suites = [
    (Suite("clean_orders")
     .not_null("order_id").unique("order_id")
     .not_null("customer_id")
     .in_set("order_status", ORDER_STATUSES)
     .dtype("order_purchase_timestamp", "datetime")
     .dtype("order_delivered_customer_date", "datetime")
     .not_null("order_purchase_timestamp")
     .between("delivery_days", 0, 400, allow_null=True)
     .custom("статус delivered ⇒ есть дата доставки",
             lambda df: df["is_delivered"] & df["order_delivered_customer_date"].isna())
     .custom("не delivered ⇒ даты доставки нет",
             lambda df: ~df["is_delivered"] & df["order_delivered_customer_date"].notna()),
     clean_orders),

    (Suite("clean_payments")
     .not_null("order_id").unique(["order_id", "payment_sequential"])
     .in_set("payment_type", PAYMENT_TYPES)
     .between("payment_value", 0.01, None, allow_null=False)
     .between("payment_installments", 0, 30)
     .foreign_key("order_id", clean_orders, "order_id", severity=WARNING),
     clean_payments),

    (Suite("clean_reviews")
     .not_null("order_id").unique("order_id")
     .between("review_score", 1, 5, allow_null=False)
     .dtype("review_creation_date", "datetime")
     .foreign_key("order_id", clean_orders, "order_id", severity=WARNING),
     clean_reviews),

    (Suite("clean_items")
     .unique(["order_id", "order_item_id"])
     .between("price", 0.01, None, allow_null=False)
     .between("freight_value", 0, None, allow_null=False)
     .foreign_key("product_id", clean_products, "product_id", severity=WARNING)
     .foreign_key("seller_id", clean_sellers, "seller_id"),
     clean_items),

    (Suite("clean_customers")
     .not_null("customer_id").unique("customer_id")
     .custom("город в нижнем регистре и без пробелов по краям",
             lambda df: df["customer_city"] != df["customer_city"].str.strip().str.lower()),
     clean_customers),
]

reports = [s.run(df).show() for s, df in suites]
report_path = save_reports(reports, paths.QUALITY / "lab03_report.json")
print(f"\nотчёт сохранён: {report_path.relative_to(paths.ROOT)}")
print(f"критических нарушений всего: {sum(len(r.critical_failed) for r in reports)}")

Проверки со статусом `warning` показывают: ссылочная целостность
платежей и отзывов относительно `clean_orders` может быть нарушена – часть
заказов уехала в карантин, и платежи по ним остались «висеть». Это не повод
останавливать обработку, но повод знать об этом. В следующей работе мы решим, что с такими
строками делать при построении модели данных.

## Та же задача в другой системе

Сеть лабораторий загружает результаты анализов из 12 приборов разных
производителей. Каждый прибор раз в час выгружает файл: у одних CSV,
у других XML, у третьих – JSON через локальный API. Результаты попадают
в систему, врачи видят их в личном кабинете пациента, а раз в сутки
считается статистика по загрузке лабораторий.

Особенности:

* единицы измерения у приборов разные (ммоль/л и мг/дл);
* прибор иногда выгружает один и тот же анализ повторно, с уточнённым результатом;
* ошибка в данных здесь – это не «неточный отчёт», а неверный результат
  анализа у конкретного человека;
* производитель может обновить прошивку и поменять формат выгрузки.

Ответьте письменно:

1. Какие источники и какого типа данные?
2. Пакетная обработка или потоковая? Что здесь определяет ответ?
3. ETL или ELT? Почему?
4. Какое зерно у таблицы результатов и что здесь считать дубликатом?
5. Три критических правила качества и что делать с их нарушителями:
   карантин, остановка обработки или предупреждение?
6. Что должно произойти, когда производитель поменял формат выгрузки?
7. Какие компоненты архитектуры здесь лишние?

### Ваш ответ

1. Источники и типы данных: …
2. Пакетная обработка или потоковая: …
3. ETL или ELT: …
4. Зерно и дубликаты: …
5. Критические правила и реакция на нарушения: …
6. Реакция на смену формата: …
7. Лишние компоненты: …

In [ ]:
lab03_result()

## Вопросы для проверки себя

1. Почему обработку, которая сначала грузит данные в хранилище, а потом
   преобразует их запросами, называют ELT, а не ETL? Назовите ситуацию,
   в которой так делать нельзя.
2. В колонке `order_delivered_customer_date` 3% пропусков. Какие вопросы надо
   задать, прежде чем решать, что с ними делать? Приведите два случая, когда
   правильные действия будут прямо противоположными.
3. Что такое дубликат, если зерно таблицы не определено? Придумайте пример,
   где одни и те же строки являются дубликатами для одной витрины и не
   являются – для другой.
4. Чем карантин лучше простого удаления строк? Назовите ситуацию, когда
   карантин, наоборот, хуже, чем остановка всей обработки.
5. Проверка «сумма `payment_value` равна сумме заказа» не прошла для 2%
   заказов. Это дефект данных или нормальная бизнес-ситуация? Как вы будете
   это выяснять и что сделаете с этими строками в промежутке?

## Что дальше

У нас появился CLEAN-слой: схема явна, плохая строка не пропадает без следа,
а дубликат определяется через зерно. Но аналитик всё ещё должен соединять пять
таблиц для каждого вопроса – и каждый может сделать это по-своему. В следующей работе
соберём факты, измерения и витрины, в которых эти определения уже зафиксированы.